In [4]:
import pandas as pd
import numpy as np
from src_RF_DT import *
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
from sklearn.metrics import roc_auc_score
from sklearn.metrics import RocCurveDisplay
from matplotlib import pyplot as plt

# 1.0 - Classificação do Desempenho Baseado em Fatores Socioeconômicos Usando Random Forest

In [3]:
colunas = ['Q001','Q002','Q003', 'Q004', 'Q005', 'Q006', 'Q007', 'Q008', 'Q009', 'Q010', 'Q011', 'Q012', 'Q013', 'Q014', 'Q015', 'Q016', 'Q017', 'Q018', 
           'Q019', 'Q020', 'Q021', 'Q022', 'Q023', 'Q024', 'Q025', 'TP_PRESENCA_LC', 'TP_PRESENCA_CH', 'TP_PRESENCA_CN', 'TP_PRESENCA_MT', 
           'TP_FAIXA_ETARIA', 'TP_ESTADO_CIVIL', 'TP_ESCOLA', 'TP_ST_CONCLUSAO', 'IN_TREINEIRO', 
           'NU_ANO', 'TP_LOCALIZACAO_ESC','TP_SIT_FUNC_ESC', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'TP_DEPENDENCIA_ADM_ESC']

df = pd.read_parquet("enem_parquet", columns = colunas)

## 1.1 - Pré-Processamento dos Dados 

In [5]:
df = pre_processor_rf_dt(df, objetivo = 'Desempenho', n_samples = 150_000)

## 1.2 - Construção da Matriz X e Vetor y

In [6]:
X = df.drop(columns=['MEDIA', 'FALTOU'])

y_media = df['MEDIA']

## 1.3 - Separação em Dados de Treino e Teste

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y_media, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [8]:
quantil = y_train.quantile(0.5)
y_train = (y_train >= quantil).astype(int)
y_val   = (y_val   >= quantil).astype(int)
y_test  = (y_test  >= quantil).astype(int)

## 1.4 - Treinando o modelo

In [9]:
rf = RandomForestClassifier()

rf.fit(X_train, y_train)

print('Ein:  %0.4f' % (1 - accuracy_score(y_train, rf.predict(X_train))))
print('Eval: %0.4f' % (1 - accuracy_score(y_val,  rf.predict(X_val))))

print(classification_report(y_val, rf.predict(X_val)))

Ein:  0.0127
Eval: 0.3181
              precision    recall  f1-score   support

           0       0.67      0.69      0.68      7379
           1       0.69      0.67      0.68      7589

    accuracy                           0.68     14968
   macro avg       0.68      0.68      0.68     14968
weighted avg       0.68      0.68      0.68     14968



## 1.5 Treinando com os Melhores Parâmetros

In [11]:
cv_rf = tune_random_forest(X_train, y_train, n_iter=30, cv=5, scoring='accuracy', random_state=42)

print('Ein:  %0.4f' % (1 - accuracy_score(y_train, cv_rf.predict(X_train))))
print('Eout: %0.4f' % (1 - accuracy_score(y_val,  cv_rf.predict(X_val))))

print(classification_report(y_val, cv_rf.predict(X_val)))

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Ein:  0.2669
Eout: 0.2943
              precision    recall  f1-score   support

           0       0.69      0.73      0.71      7379
           1       0.72      0.68      0.70      7589

    accuracy                           0.71     14968
   macro avg       0.71      0.71      0.71     14968
weighted avg       0.71      0.71      0.71     14968



In [ ]:
print(cv_rf.best_estimator_)

RandomForestClassifier(class_weight='balanced', max_depth=10,
                       min_samples_leaf=10, min_samples_split=50,
                       n_estimators=90, random_state=42)


## 1.6 - Treinando com todos os dados de treino

In [ ]:
rf = RandomForestClassifier(**cv_rf.best_params_, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y_media, test_size=0.2, random_state=42)

quantil = y_train.quantile(0.5)
y_train = (y_train >= quantil).astype(int)
y_test  = (y_test  >= quantil).astype(int)

rf.fit(X_train, y_train)

print('Ein:  %0.4f' % (1 - accuracy_score(y_train, rf.predict(X_train))))
print('Eout: %0.4f' % (1 - accuracy_score(y_test,  rf.predict(X_test))))

print(classification_report(y_test, rf.predict(X_test)))

joblib.dump(rf, 'models/rf_desempenho.pkl')

Ein:  0.2727
Eout: 0.3000
              precision    recall  f1-score   support

           0       0.69      0.76      0.72      1136
           1       0.71      0.63      0.67      1054

    accuracy                           0.70      2190
   macro avg       0.70      0.70      0.70      2190
weighted avg       0.70      0.70      0.70      2190



['models/rf_desempenho.pkl']

In [12]:
def tune_random_forest(x_train, y_train, n_iter=10, cv=5, scoring='f1_weighted', random_state=42):
    
    n_estimators = [int(x) for x in np.linspace(start=50, stop=200, num=10)]
    max_features = ['sqrt', 'log2']
    max_depth = [int(x) for x in np.linspace(start=10, stop=40, num=4)]
    max_depth.append(None)
    
    param_grid = {
    'n_estimators':      [100, 200, 300],
    'max_features':      ['sqrt', 'log2', 0.3],  # adiciona fração
    'max_depth':         [5, 10, 15, 20],         # valores menores
    'min_samples_split': [20, 50, 100],            # mais restritivo
    'min_samples_leaf':  [20, 50, 100],            # mais restritivo
    'max_samples':       [0.6, 0.8, 1.0],         # subsampling de linhas
}

    rf = RandomForestClassifier(class_weight='balanced', random_state=random_state)
    
    cv_rf = RandomizedSearchCV(      # ← mudou
        estimator=rf,
        param_distributions=param_grid,  # ← mudou
        n_iter=n_iter,               # ← agora usa o parâmetro
        cv=cv,
        scoring=scoring,
        verbose=2,
        n_jobs=-1,
        random_state=random_state    # ← reprodutibilidade
    )

    cv_rf.fit(x_train, y_train)

    return cv_rf


cv_rf = tune_random_forest(X_train, y_train, n_iter=30, cv=5, scoring='f1_weighted', random_state=42)

print(cv_rf.best_estimator_)

print('Ein:  %0.4f' % (1 - accuracy_score(y_train, cv_rf.predict(X_train))))
print('Eout: %0.4f' % (1 - accuracy_score(y_val,  cv_rf.predict(X_val))))

print(classification_report(y_val, cv_rf.predict(X_val)))

Fitting 5 folds for each of 30 candidates, totalling 150 fits
RandomForestClassifier(class_weight='balanced', max_depth=20,
                       max_features='log2', max_samples=1.0,
                       min_samples_leaf=20, min_samples_split=50,
                       random_state=42)
Ein:  0.2753
Eout: 0.2937
              precision    recall  f1-score   support

           0       0.69      0.74      0.71      7379
           1       0.72      0.68      0.70      7589

    accuracy                           0.71     14968
   macro avg       0.71      0.71      0.71     14968
weighted avg       0.71      0.71      0.71     14968



In [13]:
rf = RandomForestClassifier(**cv_rf.best_params_, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y_media, test_size=0.2, random_state=42)

quantil = y_train.quantile(0.5)
y_train = (y_train >= quantil).astype(int)
y_test  = (y_test  >= quantil).astype(int)

rf.fit(X_train, y_train)

print('Ein:  %0.4f' % (1 - accuracy_score(y_train, rf.predict(X_train))))
print('Eout: %0.4f' % (1 - accuracy_score(y_test,  rf.predict(X_test))))

print(classification_report(y_test, rf.predict(X_test)))

#joblib.dump(rf, 'models/rf_desempenho.pkl')

Ein:  0.2754
Eout: 0.2935
              precision    recall  f1-score   support

           0       0.70      0.74      0.72      9384
           1       0.72      0.68      0.70      9326

    accuracy                           0.71     18710
   macro avg       0.71      0.71      0.71     18710
weighted avg       0.71      0.71      0.71     18710

